# Parallel Progamming and Performance Engineering

This course has >80% new content. It is an evolution of my Parallel Computing course. Old introduction:

* Modern computer hardware is parallel at every scale:
  * **Instruction-level parallelism** -- a single processor core executes tens to
    hundreds of instructions at once
    * pipelines overlap the execution of many instructions across cycles
    * vector units apply the same instruction to a vector of data (SIMD)
  * **Multi-core parallelism** -- a single "processor" is many independent cores
  * **Multi-processor parallelism (NUMA)** -- many chips integrated into one machine
  * **Distributed parallelism** -- many machines connected over a network (cloud
    computing, supercomputing)
* Why bother exploiting it? Good utilization buys you:
  * **Energy efficiency** -- operating power is roughly fixed, so more work per watt
  * **Cost efficiency** -- hardware is a fixed cost, so more work per dollar
  * **Scalability** -- solve bigger problems as core counts, socket counts, and node
    counts grow

Save power, save money, solve harder problems. Parallel computing is the technology that unlocked modern AI.

Claude added:

   > But parallelism only pays off on top of code that already uses a *single* core
  well -- a badly-written serial kernel parallelizes into a badly-written parallel
  kernel. That's why this course starts with the processor and the memory hierarchy
  before it gets to multiple cores.

It's a good point. ILP is parallelism at the single core level and certain topics (pipeline and vectorization) fall in this category. But, limiting the discussion to parallelism takes too narrow a view. We will look at optimizations that are not parallel by nature, but enhance instruction throughput.  Thus, **performance engineering**.  

I stole/borrowed this term from the MIT Course "Software Performance Engineering". The 2018 version is available on line: https://ocw.mit.edu/courses/6-172-performance-engineering-of-software-systems-fall-2018/.

I've tried to develop a less hardcore treatment of the same topics that is pragmatic and designed with AI as a partner.

## Claude's summary

This course is about writing code that uses the machine well -- not just code that is
correct. Two programs can compute the same answer and differ in speed by 10x, 100x, or
more, and the difference almost never comes from a smarter algorithm. It comes from
understanding the processor, the memory system, and the compiler well enough to stop
getting in their way.

We'll build that understanding from the hardware up: how a processor executes
instructions, how memory is organized, what the compiler does and doesn't do for you,
and finally how to use multiple cores. Every idea gets a small, runnable example --
this is a hands-on course, not a theory course.

## Why this is hard

Correctness is a local property: a function is correct or it isn't, and you can check
it by reading the code. Performance is not local -- it depends on:

* **the processor** -- pipelining, out-of-order and speculative execution, branch
  prediction, instructions-per-cycle
* **the memory hierarchy** -- caches, cache lines, prefetching, false sharing
* **the compiler** -- what `-O2` actually does, auto-vectorization, when it can't help
  you
* **the algorithm's shape** -- how data is accessed, not just how many operations it
  does

A change that helps on one machine can hurt on another. The only way to know is to
measure -- so from lecture one, we build the habit of *timing things* rather than
guessing.

## RB's modification

We're not 'writing code'. We are programming....maybe.

Programming is a deceptive word at present. You will write almost no code, unless you want to. AI will generate almost all the code. So programming becomes the process of supervising code generation from a model. This has a lot of merit. In past versions of this course, the complexity of PL, tools, assembly code, syntax was a huge barrier. For me, this is a huge opportunity. We are going to do cover the same set of topics at a level of depth and complexity that was unimaginable when I last taught this course in 2024.

This course is all about the concepts. We will use multiple languages (C, C++, Python, Java, R, Rust, Assembly) and many tools and frameworks (OpenMP, Cilk, intrinsics, CUDA). You do not need to know any of these. You will need to be able to read and understand the generated code and AI is a good partner in doing so.

You will be responsible for understanding AI's implementation as it relates to the concept or learning goal for course examples and homework

(Anecdote about RB and learning Rust.)

## (Informal) AI Course Policy

Use any tool that you want at any time unless it is specifically prohibited or restricted. There will be exams and quizzes in which you will to provide written answers (your own words) and pseudo-code. All homework is designed to be done with an AI partner. See the syllabus for the whole deal.

### Expectations and Comments
  * you have access to a laptop during class
  * you have access to a reasonable AI system (HopGPT is adequate)
  * course material is designed around Claude Code (you don't need Fable. Even Sonnet 4.6 is good enough.)
  * it is possible to do all work with a ChatBot. It just makes the workflow clunkier.

## Two computations

I have said semi-seriously that
  
  >There are really only two computations in all of computer science: sorting and matrix multiplication

Nearly everything else -- searching, joining, indexing, filtering, machine learning
kernels -- reduces to one of these, or to moving data so that one of these can run
efficiently. 

These will be the culminating examples of two parts of this class:

  * Sorting: realizing the best performance on a single core will integrate:
    * Loop optimizations (unrolling)
    * Branch optimizations (branch prediction/branch-free code)
    * Vectorization/SIMD
    * ILP (out-of-order execution, dependency chains in comparison/swap sequences)
    * Pipelining (branch misprediction stalls in recursive comparison sorts)
    * Cache hierarchy (memory access patterns at larger array sizes)
    * Compiler optimization levels (function-call/frame overhead, inlining)

  * Martrix multiplication: realizing multi-core (and GPU?) parallelism
    * parallel frameworks (Cilk and OpenMP)
    * threading
    * roofline
    * loop tiling and cache blocking

If one fully understand these two example they are pretty close to an expert. But, there's a lot there: e.g. https://siboehm.com/articles/22/Fast-MMM-on-CPU.

## A Comment on the CS Curriculum

The undergraduate CS curriculum you've come up through was built on a serial model of
computation, and that model has quietly shaped almost everything you think you know
about performance. Computational complexity counts serial instructions -- an O(n log n)
algorithm is "better" than an O(n<sup>2</sup>) one because it does fewer steps, full
stop. Most programming languages, at the source level, express one instruction after
another, in order, as if that were simply what a computer does. And the systems and
architecture material that does get taught tends to build on the Von Neumann machine:
one CPU, one memory, one instruction executed to completion before the next begins.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/84/Von_Neumann_architecture.svg/500px-Von_Neumann_architecture.svg.png" width="386" title="Von Neumann Architecture" />

These are not wrong ideas -- they're powerful, and they're what you've spent years
internalizing. But they describe a machine that hasn't existed in decades, and treating
them as the whole picture is worse than incomplete; it's actively misleading. A real
processor pipelines instructions, executes them out of order, speculates past branches
before it knows the answer, and packs vector units that operate on many data elements
per instruction. None of that shows up in an O(n log n) bound, and none of it is
expressible in the "do this, then this" model most languages hand you. Two algorithms
with identical asymptotic complexity can differ in wall-clock time by an order of
magnitude because one of them cooperates with the pipeline and the cache and the other
doesn't -- you'll see exactly this in the sorting examples later, where the
"worse" O(n<sup>2</sup>) insertion sort beats the "better" O(n log n) quicksort on small
inputs, for reasons the complexity model has nothing to say about.

For decades that serial model was a reasonably safe fiction, because clock speed did
most of the work for you: wait a generation, buy a faster chip, and every program got
faster without changing a line of code. That ended in the mid-2000s -- power density
and heat dissipation put a hard ceiling on clock frequency (the "power wall"), and
chipmakers pivoted from making one core faster to putting more cores on the die. Every
laptop and phone since has been a multicore machine, and the free lunch of automatic
serial speedup ended with it -- a program that only uses one core leaves most of the
hardware idle, no matter how many transistors that hardware has. Big-O notation has
nothing to say about this transition: it counts total work, not how that work is
distributed across cores, so two algorithms with identical serial complexity can have
wildly different parallel scalability, and an asymptotically "worse" algorithm with more
exploitable parallelism can beat a "better" one on real, many-core hardware. That gap --
between how much work an algorithm does and how well that work spreads across cores --
is exactly what the second half of this course, and the work/span thinking behind
Amdahl's Law, is built to address.

## Modern Processors

### Before 1979

<img src="https://static.righto.com/images/8086-prefetch8088/die-labeled-w600.jpg" width="386" title="Intel 8088 die, 1979" />

1979 IBM 8088: a processor die looks like the Von Neumann diagram -- one control unit, one ALU, one instruction after another

### After 2025


<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/03/Ryzen_5-2600.jpg/500px-Ryzen_5-2600.jpg" width="386" title="Die shot, AMD Ryzen 5 2600 (6 cores)" />

Ryzen 5 2600:
* One chip, multiple independent cores. This die has 6, but up to 96 in server chips.

|  | 1979 (Intel 8088 + 8087 FPU) | Today (Ryzen 5 2600, 2018) |
|---|---|---|
| Transistors | 94,000 (29K CPU + 65K FPU) | ~4.8 billion |
| Clock | 5 MHz | 3.4 GHz base / 3.9 GHz boost |
| Integer ops | 0.33 MIPS | ~13.6 Gops/s single-core, ~81.6 Gops/s aggregate (4 ALU/cycle x 6 cores, base clock) |
| FLOPS | ~50,000 (via 8087 coprocessor) | ~54 GFLOPS/s single-core, ~326 GFLOPS/s aggregate (2x128-bit FMA/cycle, base clock) |

* ~51,000x the transistors, ~680x the clock, ~41,000x the integer throughput, ~1.08 million x the FLOPS \
  
Modern processors embed a couple of kinds of parallelism within each core:

### Pipeline

<img src="https://upload.wikimedia.org/wikipedia/commons/c/cb/Pipeline%2C_4_stage.svg" width="386" title="A 4-stage pipeline (Wikipedia)" />

* One instruction, several stages (fetch, decode, execute, memory, writeback)
* Independent instructions overlap stages -- a new one starts, one retires, every cycle
* Requires independence: instructions can't overlap if one needs another's result

<img src="https://upload.wikimedia.org/wikipedia/commons/6/67/Pipeline%2C_4_stage_with_bubble.svg" width="386" title="A pipeline stall (Wikipedia)" />

* Dependent instruction -> **stall**: bubble cycles instead of useful work
* Complexity analysis counts instructions, not dependencies between them -- see [pipeline/](../examples/pipeline/)

### Vector

<img src="https://upload.wikimedia.org/wikipedia/commons/3/3b/Visual_representation_of_the_SIMD_instruction.svg" width="386" title="A SIMD instruction (Wikipedia)" />

* One instruction, multiple data elements packed into one register (SIMD)
* e.g. one vector add = 4/8/16 scalar adds, same latency as one
* Reachable from source: auto-vectorization, or intrinsics directly -- see [vectorization/](../examples/vectorization/)
* Why the SIMD sort variants (and the roofline matmul kernel) exist later in the course

### How much parallelism is available?

    core_count x pipeline_depth x vector_width

* Typically 200-1000 instructions in flight at once
* None of it is visible in an algorithm's asymptotic complexity
* Our job: feed the processor enough independent work to use it
  * **implicitly** -- write code patterns the compiler pipelines/vectorizes cleanly
  * **explicitly** -- multithreading (OpenMP, Cilk), vector intrinsics